In [1]:
import numpy as np
import resource
from collections import defaultdict
from transformers import AutoTokenizer
from transduction.lm.statelm import HfTokenizerVocab
from transduction.lm.ngram import CharNgramLM
from transduction.lm.fused_transduced import FusedTransducedLM
from transduction.lm.character_beam import CharacterBeam
from transduction.util import Timeout, timelimit, set_memory_limit
set_memory_limit(12)

In [2]:
tokenizer = AutoTokenizer.from_pretrained('gpt2', use_fast=False, local_files_only=True)
hf_vocab = HfTokenizerVocab(tokenizer)
_decode = hf_vocab.decode
#drop = {x.encode() for x in tokenizer.all_special_tokens}
#all_token_ids = sorted(i for i in range(len(_decode)) if _decode[i] not in drop)

eos_bytes = tokenizer.eos_token.encode()
eos_id = hf_vocab.encode[eos_bytes]

In [23]:
train_sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "A stitch in time saves nine.",
    "To be or not to be, that is the question.",
    "All that glitters is not gold.",
    "Actions speak louder than words.",
    "Practice makes perfect.",
    "Where there is a will, there is a way.",
] * 3
train_ids = [tokenizer.encode(s) for s in train_sentences]
train_used = sorted(set(tid for seq in train_ids for tid in seq))
lm = CharNgramLM.train(train_ids, n=5, alpha=0.01, alphabet=set(range(len(_decode))))

In [24]:
c = CharacterBeam(lm, dict(enumerate(_decode)), K=10, eos_token=eos_id)

In [25]:
#lm.initial().logp_next

In [27]:
x = b'The quick' 
x + bytes(c(x).greedy_decode(100))

b'The quick statistically there statistically there statistically there statistically there statistically there'